# Universal LLM API - All-models validation
Validate the actual deployed model surface: discovery, chat, numeric embeddings, and
stored Responses create/get/input-items. Complete Environment Setup and Backend
Onboarding first. Requires deployment permission and an annotated capability map.
Image-only models are explicitly reported and tested in Image Models.

The access policy below intentionally leaves `allowedModels` unrestricted. Configuration
metadata plans coverage; authenticated gateway requests establish whether it works.

## 1. Configure

Explicit overrides win over azd. Operations are inferred from model names by default; explicit overrides replace the assumptions.

Configure this notebook's environment, explicit feature opt-ins, and run-owned resources without deploying anything.

**Input:** The selected repository kernel; edit the configuration values below and provide required secrets through environment variables.

**Output:** Configuration globals for the following initialization step; no cloud deployment or test is performed.


### Automatic models and optional overrides

Leave `environment=None`, azd enabled, `operation_overrides=None`, and
`skipped_models=None` to use the selected deployment without editing model lists.
Operations default to name-based **assumptions**, verified by real requests:
`image`/`flux` → image generation/edit; otherwise `embedding`/`text` → embeddings;
otherwise `gpt` → chat and Responses; other names → chat. Matching ignores case.

Optional examples (replace example names with names in the loaded inventory):

```python
# Replace operations only for these models; other models retain automatic mapping.
operation_overrides = {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
# Use operation_overrides = {} to discard stored azd operation overrides.
# Disable one model's tests with an empty override: {"my-model": []}.
skipped_models = ["mini", "legacy"]  # Case-insensitive substrings; backend deployment is unchanged.
overrides = {"resource_group": "rg-other-hub", "apim_name": "apim-other-hub"}  # Explicit azd-value overrides.
# To require a complete manually mapped catalogue: overrides["infer_operations"] = False.
```

Streaming is assumed for inferred chat models. Explicit mappings need `streaming`
when that scenario is selected. Responses GET/input-items returning HTTP 404 are warnings/SKIP for every model,
including explicit overrides, not successful retrieval. Other unsupported-operation
exceptions remain limited to inferred GPT-mini models; explicit mappings require
those non-404 retrieval checks. Authentication, throttling, and server failures
are never silently accepted.


In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(root / "shared"))
import utils
with utils.notebook_step('1. Configure', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='Configuration globals for the following initialization step; no cloud deployment or test is performed.', namespace=globals()):
    from pathlib import Path
    import sys, json, time, uuid
    import requests
    from validationtools import ValidationContext, ValidationRun
    from contracttools import write_parameters, deploy_contract, access_policy
    from apimtools import APIMClientTool
    from consumerhelpers import product_exists

    environment = None  # None uses the selected azd environment.
    from_azd = True
    overrides = {}  # Explicit resource coordinates/model capabilities win over azd.
    model_override = None
    cleanup_enabled = False

    scenario = "universal-all-models"

    # None uses azd operation overrides when present, otherwise automatic name-based mapping.
    # Example: {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
    # An empty list for one model disables its tests; explicit lists replace that model's inferred operations.
    # A supplied dictionary replaces azd's operation map; {} resets it to automatic mapping.
    operation_overrides = None
    # None uses VALIDATION_SKIPPED_MODELS from azd (default []); matches substrings without case sensitivity.
    # Example: ["mini", "legacy"] skips any model containing either keyword in validation, not deployment.
    skipped_models = None


## 2. Initialize and preflight

Load the selected azd deployment with inferred operations and optional case-insensitive skip keywords. Image-only models in a mixed inventory are out of scope for this text-operation notebook, not missing inputs.

**Input:** Configuration from the previous cell, resource coordinates, inferred or explicitly overridden model operations, and enabled integration prerequisites.

**Output:** Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.


In [ ]:
with utils.notebook_step('2. Initialize and preflight', inputs={'scenario': globals().get('scenario'), 'overrides': globals().get('overrides'), 'environment': globals().get('environment'), 'from_azd': globals().get('from_azd')}, expected='Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.', namespace=globals()):
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=from_azd, operation_overrides=operation_overrides, skipped_models=skipped_models)
    mandatory_inputs = {
        "gateway.subscription_id": context.config.get("subscription_id"),
        "gateway.resource_group": context.config.get("resource_group"),
        "gateway.location": context.config.get("location"),
        "gateway.apim_name": context.config.get("apim_name"),
    }
    utils.require_inputs(mandatory_inputs)
    utils.require_capabilities(context.catalog)
    catalog = context.catalog
    # Mixed inventories are valid: unsupported operation families are skipped in the test plan.
    apim = APIMClientTool(context.config["resource_group"], context.config.get("apim_name") or "")
    apim.initialize_context(context.account)
    location = context.config["location"]
    run_id = uuid.uuid4().hex[:12]
    run_dir = root / "validation" / ".runs" / scenario / run_id
    run = ValidationRun(scenario)
    apim_coordinates = {"subscriptionId": context.config["subscription_id"],
                        "resourceGroupName": context.config["resource_group"], "name": apim.apim_resource_name}
    utils.format_output('2. Initialize and preflight', {'value_1': "Environment:", 'value_2': environment or "selected azd environment"}, status='info', part='RESULT')
    utils.format_output('2. Initialize and preflight', {'value_1': "Gateway:", 'value_2': apim.apim_resource_gateway_url}, status='info', part='RESULT')
    utils.format_output('2. Initialize and preflight', {'value_1': "Artifacts:", 'run_dir': run_dir}, status='info', part='RESULT')

    run_dir.mkdir(parents=True, exist_ok=True)


## 3. Preview the access contract

The generated `.bicepparam` is the deployment input. Inspect or edit it before continuing.

Generate the editable parameter artifact for 3. Preview the access contract and display the proposed configuration without deploying it.

**Input:** `apim_coordinates`, `run_id`, `run_dir`, `scenario`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [ ]:
with utils.notebook_step('3. Preview the access contract', inputs={'apim_coordinates': globals().get('apim_coordinates'), 'run_id': globals().get('run_id'), 'run_dir': globals().get('run_dir'), 'scenario': globals().get('scenario')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    business_unit, use_case, contract_environment = "Validation", scenario.replace("-", "")[:24], "DEV"
    product_id = f"LLM-{business_unit}-{use_case}-{contract_environment}"
    policy_xml = access_policy([], 100000, 1000000, "Hourly")
    parameters = {
        "apim": apim_coordinates, "keyVault": apim_coordinates,
        "useTargetAzureKeyVault": False,
        "useCase": {"businessUnit": business_unit, "useCaseName": use_case, "environment": contract_environment},
        "apiNameMapping": {"LLM": ["universal-llm-api", "azure-openai-api", "unified-ai-api"]},
        "services": [{"code": "LLM", "apiKeySecretName": product_id + "-KEY",
                      "endpointSecretName": product_id + "-ENDPOINT", "policyXml": policy_xml}],
    }
    access_params = write_parameters(root / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep",
                                     parameters, run_dir / "access")
    utils.format_output('3. Preview the access contract', {'policy_xml': policy_xml}, status='info', part='RESULT')
    utils.format_output('3. Preview the access contract', {'value_1': access_params.read_text(encoding="utf-8")}, status='info', part='RESULT')


## 4. Deploy and retrieve the exact contract credential

Deploy the reviewed artifact for 4. Deploy and retrieve the exact contract credential, then inspect its real resource outputs and ownership.

**Input:** `product_id`, `access_params`, `location`, `scenario`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('4. Deploy and retrieve the exact contract credential', inputs={'product_id': globals().get('product_id'), 'access_params': globals().get('access_params'), 'location': globals().get('location'), 'scenario': globals().get('scenario')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    expected_product_id = product_id
    product_preexists = product_exists(context, apim, product_id)
    owned_product_id = None
    access_outputs = deploy_contract(access_params, location, f"validation-{scenario}-access")
    deployed_endpoints = access_outputs.get("endpoints", {}).get("value", [])
    if len(deployed_endpoints) != 1 or deployed_endpoints[0].get("productId") != expected_product_id:
        raise RuntimeError("Expected exactly one endpoint for the configured product; review parameter edits.")
    product_id = deployed_endpoints[0]["productId"]
    if not product_preexists:
        owned_product_id = product_id
    api_key = apim.get_subscription_key(deployed_endpoints[0]["subscriptionName"])
    run.check("access-contract-deployed", bool(access_outputs), expected="Nonempty deployment outputs")


## 5. Discover the actual gateway model set

Exercise 5. Discover the actual gateway model set and compare the actual response/evidence with the expected feature behavior.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('5. Discover the actual gateway model set', inputs={}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    models_api = apim.discover_api_exact("models")["url"]
    headers = {"api-key": api_key, "Content-Type": "application/json"}
    response = utils.http_call(requests.get, f"{models_api}/models", headers=headers, timeout=30)
    response.raise_for_status()
    body = response.json()
    items = body.get("data", body.get("value"))
    if not isinstance(items, list) or not items:
        raise RuntimeError("Discovery returned no model entries.")
    if any(not isinstance(item, dict) or not isinstance(item.get("name") or item.get("id"), str)
           or not (item.get("name") or item.get("id")) for item in items):
        raise RuntimeError("Each discovery entry must contain a nonempty model name or ID.")
    discovered_models = [item.get("name") or item.get("id") for item in items]
    discovered_aliases = {item.get("name") or item.get("id") for item in items
                          if item.get("type") == "alias"
                          or item.get("properties", {}).get("model", {}).get("format") == "Alias"}
    configured_models = {m["name"] if isinstance(m, dict) else m
                         for b in context.config["llm_backends"] for m in b["supportedModels"]}
    utils.format_output('5. Discover the actual gateway model set', {'value_1': "Configured but not discovered:", 'value_2': sorted(configured_models - set(discovered_models))}, status='info', part='RESULT')
    utils.format_output('5. Discover the actual gateway model set', {'value_1': "Discovered but not configured:", 'value_2': sorted(set(discovered_models) - configured_models)}, status='info', part='RESULT')
    run.check("model-discovery", configured_models.issubset(discovered_models),
              expected="All configured models visible to unrestricted contract", actual=discovered_models)


## 6. Plan capability-specific operations

Prepare the 6. Plan capability-specific operations state shown below for its subsequent feature-specific requests and assertions.

**Input:** `discovered_models`, `discovered_aliases`

**Output:** `test_plan`, `model`, `operations`, `selected` are available to the following cells.


In [ ]:
with utils.notebook_step('6. Plan capability-specific operations', inputs={'discovered_models': globals().get('discovered_models'), 'discovered_aliases': globals().get('discovered_aliases')}, expected='`test_plan`, `model`, `operations`, `selected` are available to the following cells.', namespace=globals()):
    test_plan = []
    for model in discovered_models:
        if catalog.is_skipped(model):
            run.skip(f"model:{model}", catalog.skip_reason(model))
            continue
        if model in discovered_aliases:
            run.skip(f"alias:{model}", "Alias strategy and member routing are covered in Model Aliases.")
            continue
        operations = catalog.operations(model)
        selected = [op for op in operations if op in ("chat", "embeddings", "responses")]
        if not operations:
            run.blocked(f"capabilities:{model}", "Set VALIDATION_MODEL_CAPABILITIES for this deployed model.")
        elif not selected:
            run.skip(f"operations:{model}", "Non-text operation; use Image Models or its capability-specific notebook.")
        else:
            test_plan.extend((model, op) for op in selected)
    utils.format_output('6. Plan capability-specific operations', {'value_1': "Planned model/operation pairs:", 'test_plan': test_plan}, status='info', part='RESULT')


## 7. Execute

Each model gets only its declared operations. Responses GET and input-items GET returning HTTP 404 are warnings/SKIP for every model, not errors or successful retrieval. Other failures retain the documented operation-specific rules.

Exercise 7. Execute and compare the actual response/evidence with the expected feature behavior.

**Input:** `test_plan`, `headers`, `model`, `response`, `models_api`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('7. Execute', inputs={'test_plan': globals().get('test_plan'), 'headers': globals().get('headers'), 'model': globals().get('model'), 'response': globals().get('response'), 'models_api': globals().get('models_api')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    with requests.Session() as session:
        session.headers.update(headers)
        for model, operation in test_plan:
            case = f"{model}:{operation}"
            try:
                if operation == "chat":
                    response = utils.http_call(session.post, f"{models_api}/chat/completions",
                        json={"model": model, "messages": [{"role": "user", "content": "What is 2+2? Reply briefly."}]},
                        timeout=90)
                    response.raise_for_status()
                    data = response.json()
                    content = data.get("choices", [{}])[0].get("message", {}).get("content")
                    run.check(case, bool(content), expected="Nonempty chat content", actual=response.status_code)
                elif operation == "embeddings":
                    response = utils.http_call(session.post, f"{models_api}/embeddings",
                        json={"model": model, "input": "A short validation sentence.", "encoding_format": "float"}, timeout=90)
                    response.raise_for_status()
                    vector = response.json().get("data", [{}])[0].get("embedding")
                    run.check(case, isinstance(vector, list) and bool(vector)
                              and all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in vector),
                              expected="Nonempty numeric embedding", actual=response.status_code)
                else:
                    response = utils.http_call(session.post, f"{models_api}/responses",
                        json={"model": model, "input": "What is 2+2? Reply briefly.", "store": True}, timeout=90)
                    response.raise_for_status()
                    data = response.json()
                    response_id = data.get("id")
                    run.check(case + ":create", bool(response_id) and bool(data.get("output")),
                              expected="Stored response ID and output", actual=response.status_code)
                    if not response_id:
                        run.blocked(case + ":retrieve", "Create returned no ID.")
                        continue
                    # The same session preserves the pool affinity cookie across the lifecycle.
                    response = utils.http_call(session.get, f"{models_api}/responses/{response_id}", timeout=30)
                    try:
                        retrieval_body = response.json()
                    except ValueError:
                        retrieval_body = None
                    if catalog.optional_response_retrieval(model, response.status_code, retrieval_body):
                        run.skip(case + ":get", f"Responses GET returned HTTP {response.status_code}; retrieval not validated (warning).")
                    else:
                        run.check(case + ":get", response.status_code == 200 and isinstance(retrieval_body, dict) and retrieval_body.get("id") == response_id,
                                  expected="200 and original response ID", actual=response.status_code)
                    response = utils.http_call(session.get, f"{models_api}/responses/{response_id}/input_items", timeout=30)
                    try:
                        retrieval_body = response.json()
                    except ValueError:
                        retrieval_body = None
                    if catalog.optional_response_retrieval(model, response.status_code, retrieval_body):
                        run.skip(case + ":input-items", f"Responses input-items GET returned HTTP {response.status_code}; input-items not validated (warning).")
                    else:
                        run.check(case + ":input-items", response.status_code == 200 and isinstance(retrieval_body, dict)
                                  and bool(retrieval_body.get("data")), expected="200 and nonempty input items",
                                  actual=response.status_code)
            except (requests.RequestException, ValueError, KeyError, IndexError) as exc:
                run.check(case, False, expected="Successful operation with valid body", actual=str(exc))


## 8. Unknown-model routing rejection

Exercise 8. Unknown-model routing rejection and compare the actual response/evidence with the expected feature behavior.

**Input:** `headers`, `models_api`, `response`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('8. Unknown-model routing rejection', inputs={'headers': globals().get('headers'), 'models_api': globals().get('models_api'), 'response': globals().get('response')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    response = utils.http_call(requests.post, f"{models_api}/chat/completions", headers=headers,
        json={"model": "definitely-not-a-real-model-xyz", "messages": [{"role": "user", "content": "Hi"}]}, timeout=30)
    error = response.json().get("error", {})
    run.check("unknown-model-rejected", response.status_code in (400, 404)
              and bool(error.get("code")) and error.get("code") != "unauthorized_model_access",
              expected="Routing 400/404, not access-contract RBAC", actual=response.status_code)


## 9. Results

Collect the final 9. Results verdict without discarding earlier failed or blocked required cases.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** The strict validation summary; any failed or blocked required coverage raises.


In [ ]:
with utils.notebook_step('9. Results', inputs={}, expected='The strict validation summary; any failed or blocked required coverage raises.', namespace=globals()):
    run.finish()


## 10. Cleanup (opt-in)

Apply the explicitly enabled 10. Cleanup (opt-in) action only to resources owned by this run.

**Input:** `cleanup_enabled`, `product_id`, `owned_product_id`

**Output:** Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.


In [ ]:
with utils.notebook_step('10. Cleanup (opt-in)', inputs={'cleanup_enabled': globals().get('cleanup_enabled'), 'product_id': globals().get('product_id'), 'owned_product_id': globals().get('owned_product_id')}, expected='Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.', namespace=globals()):
    utils.format_output('10. Cleanup (opt-in)', {'value_1': "Owned product:", 'product_id': product_id}, status='info', part='RESULT')
    if cleanup_enabled and not product_preexists:
        if product_id != owned_product_id:
            raise RuntimeError("Edited parameters changed the product ID; ownership is unproven, so cleanup is blocked.")
        apim.client.product.delete(context.config["resource_group"], apim.apim_resource_name,
                                   product_id, delete_subscriptions=True, if_match="*")
        utils.format_output('10. Cleanup (opt-in)', {'value_1': "Deleted this run's access product and subscriptions. Shared backends were retained."}, status='info', part='RESULT')
    else:
        utils.format_output('10. Cleanup (opt-in)', {'value_1': "Cleanup disabled or product reused; existing resources and shared backends are retained."}, status='info', part='RESULT')
